In [12]:
!pip install playwright

  Using cached playwright-1.58.0-py3-none-win_amd64.whl.metadata (3.5 kB)
  Using cached pyee-13.0.1-py3-none-any.whl.metadata (3.0 kB)
Using cached playwright-1.58.0-py3-none-win_amd64.whl (36.8 MB)
   ---------------------------------------- 0.0/238.2 kB ? eta -:--:--
   ----------------------------- ---------- 174.1/238.2 kB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 238.2/238.2 kB 3.7 MB/s eta 0:00:00
Using cached pyee-13.0.1-py3-none-any.whl (15 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\cin_c\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
!playwright install chromium

In [1]:
%%writefile scraper_p12.py
import json
from playwright.sync_api import sync_playwright

urls = {
    "pagina12": [
        "https://www.pagina12.com.ar/825014-la-corte-suprema-de-santa-fe-reconocio-el-dano-genetico-prov/",
        "https://www.pagina12.com.ar/235451-glifosato-una-investigacion-argentina-confirma-su-peligro/",
        "https://www.pagina12.com.ar/275246-nuevo-estudio-vincula-al-glifosato-con-el-cancer-malformacio/",
    ],
    "clarin": [
        "https://www.clarin.com/rural/productores-podran-seguir-usando-glifosato-misiones_0_HQRRFpwW2j.html",
        "https://www.clarin.com/rural/soja-rr-glifosato_0_YdlYzZ4H-.html",
        "https://www.clarin.com/viste/que-es-el-glifosato-y-cual-es-su-impacto-en-la-economia-y-en-la-ecologia_0_95xysQIZlO.html",
    ]
}

resultados = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context()
    page = context.new_page()

    for grupo, lista_urls in urls.items():
        for url in lista_urls:
            print(f"Scrapeando: {url}")
            page.goto(url)
            page.wait_for_timeout(3000)

            # Cerramos modales si aparecen
            try:
                boton = page.wait_for_selector('button:has-text("Aceptar")', timeout=3000)
                if boton:
                    boton.click()
            except:
                pass

            texto = page.evaluate("() => document.body.innerText")
            resultados.append({
                "url": url,
                "grupo_comparacion": grupo,
                "texto": texto
            })

    browser.close()

with open("corpus_raw.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("Listo. Guardado en corpus_raw.json")

Overwriting scraper_p12.py


In [2]:
%%writefile scraper_glifosato.py
import json
from playwright.sync_api import sync_playwright

urls = {
    "pagina12": [
        "https://www.pagina12.com.ar/825014-la-corte-suprema-de-santa-fe-reconocio-el-dano-genetico-prov/",
        "https://www.pagina12.com.ar/235451-glifosato-una-investigacion-argentina-confirma-su-peligro/",
        "https://www.pagina12.com.ar/275246-nuevo-estudio-vincula-al-glifosato-con-el-cancer-malformacio/",
    ],
    "clarin": [
        "https://www.clarin.com/rural/productores-podran-seguir-usando-glifosato-misiones_0_HQRRFpwW2j.html",
        "https://www.clarin.com/rural/soja-rr-glifosato_0_YdlYzZ4H-.html",
        "https://www.clarin.com/viste/que-es-el-glifosato-y-cual-es-su-impacto-en-la-economia-y-en-la-ecologia_0_95xysQIZlO.html",
    ]
}

resultados = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context()
    page = context.new_page()

    for grupo, lista_urls in urls.items():
        for url in lista_urls:
            print(f"Scrapeando: {url}")
            page.goto(url)
            page.wait_for_timeout(3000)

            try:
                boton = page.wait_for_selector('button:has-text("Aceptar")', timeout=3000)
                if boton:
                    boton.click()
            except:
                pass

            texto = page.evaluate("() => document.body.innerText")
            resultados.append({
                "url": url,
                "grupo_comparacion": grupo,
                "texto": texto
            })
            print(f"OK - {len(texto)} caracteres extraídos")

    browser.close()

with open(r"C:\Users\cin_c\OneDrive\Desktop\Repositorios\pln\coronel-cintia-pln-1c-2026\007_tpi_2\003 - LAB\corpus_raw.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("Listo. Guardado en corpus_raw.json")

Overwriting scraper_glifosato.py


In [25]:
!python scraper_glifosato.py

^C


In [9]:
import json
import pandas as pd

with open("corpus_raw.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

df_raw = pd.DataFrame(datos)
df_raw.head()

,url,grupo_comparacion,texto
0,https://www.pagina12.com.ar/825014-la-corte-su...,pagina12,La Corte Suprema de Santa Fe ratificó un fallo...
1,https://www.pagina12.com.ar/235451-glifosato-u...,pagina12,"""Los resultados presentados aquí deberían ser ..."
2,https://www.pagina12.com.ar/275246-nuevo-estud...,pagina12,Existen más de 1100 estudios científicos que d...
3,https://www.clarin.com/rural/productores-podra...,clarin,ERNESTO\nAZARKEVICH\nCORRESPONSALÍA MISIONES\n...
4,https://www.clarin.com/rural/soja-rr-glifosato...,clarin,HÉCTOR A.\nHUERGO\nResumen\nCompartir\nSeguir\...


In [10]:
print(len(df_raw))
df_raw[["url", "grupo_comparacion"]]

6


,url,grupo_comparacion
0,https://www.pagina12.com.ar/825014-la-corte-su...,pagina12
1,https://www.pagina12.com.ar/235451-glifosato-u...,pagina12
2,https://www.pagina12.com.ar/275246-nuevo-estud...,pagina12
3,https://www.clarin.com/rural/productores-podra...,clarin
4,https://www.clarin.com/rural/soja-rr-glifosato...,clarin
5,https://www.clarin.com/viste/que-es-el-glifosa...,clarin
